# Spectrogram Resolution Experiment
Trains the same 2D CNN on spectrograms of decreasing resolution (n×n where n ∈ {5, 7, 10})
to find the point where classification quality degrades.
The 10×10 result is already available from `train_cnn.ipynb`; this notebook
builds the missing sizes and re-trains for each.

**Expected runtime: ~5–10 min per resolution (~15–30 min total).**

In [ ]:
import subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
EPOCHS     = 30
LR         = 1e-3
PATIENCE   = 5
N_VALUES   = [5, 7, 10]   # resolutions to compare — add more if desired

print(f"Device: {DEVICE}")
print(f"Resolutions to test: {N_VALUES}")

## Build any missing spectrogram files

In [ ]:
spec_dir = Path("../data/spectrograms")

for n in N_VALUES:
    out = spec_dir / f"spectrograms_{n}x{n}.npy"
    if out.exists():
        print(f"{out.name}  already exists, skipping.")
    else:
        print(f"Building {out.name} ...")
        t0 = time.time()
        result = subprocess.run(
            [sys.executable, "../data_code/build_spectrograms.py", "--n", str(n)],
            capture_output=True, text=True
        )
        print(result.stdout.strip())
        if result.returncode != 0:
            print("ERROR:", result.stderr.strip())
        else:
            print(f"  Done in {time.time()-t0:.1f}s")

## Model — same architecture as train_cnn.ipynb
AdaptiveAvgPool2d(2) at the end makes the model work for any input resolution ≥ 4×4.

In [ ]:
class EarthquakeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


class SpectrogramDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):        return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

## Training & evaluation loop over resolutions

In [ ]:
meta   = pd.read_csv("../data/metadata.csv", low_memory=False)
labels = meta["label"].values.astype(np.float32)
splits = meta["split"].values

results = {}   # n → {f1, accuracy, auc, epochs_run}


def train_and_eval(n):
    spec_path = spec_dir / f"spectrograms_{n}x{n}.npy"
    print(f"\n{'='*55}")
    print(f"  Resolution: {n}x{n}")
    print(f"{'='*55}")

    X_spec = np.load(spec_path)[:, np.newaxis, :, :]
    train_mask = splits == "train"
    mu  = float(X_spec[train_mask].mean())
    std = float(X_spec[train_mask].std())
    X_spec = ((X_spec - mu) / (std + 1e-8)).astype(np.float32)

    def make_loader(split_name, shuffle=False):
        mask = splits == split_name
        return DataLoader(
            SpectrogramDataset(X_spec[mask], labels[mask]),
            batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
        )

    train_loader = make_loader("train", shuffle=True)
    val_loader   = make_loader("val")
    test_loader  = make_loader("test")

    model     = EarthquakeCNN().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=3, factor=0.5
    )

    best_val_f1, patience_counter, epochs_run = 0.0, 0, 0
    ckpt = f"best_model_{n}x{n}.pt"

    try:
        for epoch in range(1, EPOCHS + 1):
            epochs_run = epoch
            model.train()
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                optimizer.zero_grad()
                criterion(model(X_batch), y_batch).backward()
                optimizer.step()

            model.eval()
            val_loss, val_probs, val_true = 0.0, [], []
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                    logits = model(X_batch)
                    val_loss += criterion(logits, y_batch).item() * len(y_batch)
                    val_probs.append(torch.sigmoid(logits).cpu().numpy())
                    val_true.append(y_batch.cpu().numpy())
            val_loss /= len(val_loader.dataset)
            val_probs = np.concatenate(val_probs)
            val_f1    = f1_score(np.concatenate(val_true), (val_probs >= 0.5).astype(int))
            scheduler.step(val_loss)

            print(f"  Epoch {epoch:2d}  val_loss={val_loss:.4f}  val_F1={val_f1:.4f}")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), ckpt)
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch}")
                    break
    except KeyboardInterrupt:
        print(f"  Stopped at epoch {epoch}")

    # ── Evaluate on test set ──────────────────────────────────────────────
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    test_probs, test_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits = model(X_batch.to(DEVICE))
            test_probs.append(torch.sigmoid(logits).cpu().numpy())
            test_true.append(y_batch.numpy())
    test_probs = np.concatenate(test_probs)
    test_true  = np.concatenate(test_true)
    test_preds = (test_probs >= 0.5).astype(int)

    res = {
        "f1":         round(f1_score(test_true, test_preds), 4),
        "accuracy":   round(accuracy_score(test_true, test_preds), 4),
        "auc":        round(roc_auc_score(test_true, test_probs), 4),
        "epochs_run": epochs_run,
    }
    print(f"  TEST  F1={res['f1']}  Acc={res['accuracy']}  AUC={res['auc']}")
    return res


for n in N_VALUES:
    results[n] = train_and_eval(n)

## Results summary

In [ ]:
print(f"\n{'n':>4}  {'F1':>8}  {'Accuracy':>10}  {'AUC-ROC':>9}  {'Epochs':>7}")
print("-" * 46)
for n in N_VALUES:
    r = results[n]
    print(f"{n:>4}x{n:<2}  {r['f1']:>8.4f}  {r['accuracy']:>10.4f}  {r['auc']:>9.4f}  {r['epochs_run']:>7}")

## F1 vs resolution plot

In [ ]:
ns   = N_VALUES
f1s  = [results[n]["f1"]  for n in ns]
aucs = [results[n]["auc"] for n in ns]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(ns, f1s, marker="o", linewidth=2)
axes[0].set_xticks(ns)
axes[0].set_xticklabels([f"{n}×{n}" for n in ns])
axes[0].set_xlabel("Spectrogram resolution")
axes[0].set_ylabel("F1 (test)")
axes[0].set_title("F1 vs resolution")
axes[0].grid(True, alpha=0.3)

axes[1].plot(ns, aucs, marker="s", color="darkorange", linewidth=2)
axes[1].set_xticks(ns)
axes[1].set_xticklabels([f"{n}×{n}" for n in ns])
axes[1].set_xlabel("Spectrogram resolution")
axes[1].set_ylabel("AUC-ROC (test)")
axes[1].set_title("AUC-ROC vs resolution")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()